This notebook showcases the use of TxConformal in the enamine real deployment of our paper (pre-experiment analysis after model selection).

In [1]:
import os
import numpy as np 
import pickle
import pandas as pd  
from sklearn.linear_model import LinearRegression

from txconformal import TxConformal
from txconformal.features.providers import FeaturesProvider
from txconformal.features.quantiles import make_bins


### Read in raw model predictions and embeddings

In [ ]:
MODEL_NAMES = ['AttentiveFP', 'CNN', 'GCN', 'GIN_AttrMasking',
               'GIN_ContextPred', 'Morgan', 'NeuralFP', 'rdkit_2d']
path = '/Users/yjinstat/Desktop/Research/collaborations/cf_drug_discovery/data/hts/classification/bce_loss/'   

# =============================================================================
# # ensemble models, predictions and weights
# =============================================================================

model_id_set = [3,4]

data_cal_pred = pd.DataFrame()
data_test_pred = pd.DataFrame()

cal_weights = pd.DataFrame({})
test_weights = pd.DataFrame({})

# collect model predictions
data_test_all = pd.read_csv("/Users/yjinstat/Desktop/Research/collaborations/cf_drug_discovery/workflow/hts_workflow/data_for_nate/diversity_novelty_filtered.csv")
data_calib_all = pd.read_csv("/Users/yjinstat/Desktop/Research/collaborations/cf_drug_discovery/workflow/hts_workflow/data_for_nate/calib_all.csv")
data_test_all.columns = ['index'] + list(data_test_all.columns[1:])

for model_id in range(8):
    test_data = pd.read_csv(path + MODEL_NAMES[model_id] + '/enamine_hts_predict_seed1_pred.csv') 
    prev_data = pd.read_csv(path + MODEL_NAMES[model_id] + '/ncb_random_seed1_pred.csv')  
    # get label and prediction
    data_cal = prev_data[(prev_data['split'] == 'calib') | (prev_data['split'] == 'test')].copy(deep=True)[['Label', 'pred']].reset_index(drop=True)
    data_test = test_data[test_data['split'] == 'test'].copy(deep=True)[['Label', 'pred']].reset_index(drop=True) 
    # attach to the giant data frame
    if data_cal_pred.shape[0] == 0:
        data_cal_pred = data_cal.copy()
        data_cal_pred['pred'+str(model_id)] = data_cal['pred']
        data_test_pred = data_test.copy() 
        data_test_pred['pred'+str(model_id)] = data_test['pred'] 
    else:
        data_cal_pred['pred'+str(model_id)] = data_cal['pred']
        data_test_pred['pred'+str(model_id)] = data_test['pred']
        data_cal_pred['pred'] = data_cal_pred['pred'] + data_cal['pred']
        data_test_pred['pred'] = data_test_pred['pred'] + data_test['pred'] 
        
data_test_pred = data_test_pred.iloc[np.array(data_test_all['index']), :]
data_test_pred = data_test_pred.reset_index()
    
# collect hidden embeddings from model_id_set
calib_feature_list = []
test_feature_list = []
    
for model_id in model_id_set:  
    prev_data = pd.read_csv(path + MODEL_NAMES[model_id] + '/ncb_random_seed1_pred.csv')  
    # read in hidden embedding
    with open(path + MODEL_NAMES[model_id] + '/enamine_hts_predict_seed1_hid.pkl', 'rb') as file:
        test_feature = pickle.load(file)
    with open(path + MODEL_NAMES[model_id] + '/ncb_random_seed1_hid.pkl', 'rb') as file:
        prev_feature = pickle.load(file)
    
    calib_feature_list.append(prev_feature[(prev_data['split'] == 'calib') | (prev_data['split'] == 'test'),:])
    test_feature_list.append(test_feature[np.array(data_test_all['index']), :])

calib_feature_all = pd.DataFrame(np.concatenate(calib_feature_list, axis=1))
test_feature_all = pd.DataFrame(np.concatenate(test_feature_list, axis=1)) 
calib_feature_all.columns = ["mdl3_"+str(x) for x in range(256)] +["mdl4_"+ str(x) for x in range(256)] 
test_feature_all.columns = ["mdl3_"+str(x) for x in range(256)] +["mdl4_"+ str(x) for x in range(256)]

data_cal_pred['pred'] = (data_cal_pred['pred3']+data_cal_pred['pred4']) / 2
data_test_pred['pred'] = (data_test_pred['pred3']+data_test_pred['pred4']) / 2
data_calib = pd.concat([data_cal_pred, pd.DataFrame(calib_feature_all)], axis=1)
data_test = pd.concat([data_test_pred, pd.DataFrame(test_feature_all)], axis=1) 
 
data_ct = pd.concat([data_calib, data_test], axis= 0).reset_index(drop=True)      

# add bins of predicted values 
cal_bin3, test_bin3, _ = make_bins(data_ct['pred3'][:data_calib.shape[0]].to_numpy(),
                                   data_ct['pred3'][data_calib.shape[0]:].to_numpy(), n_bins=10)
data_ct = pd.concat([data_ct, pd.DataFrame(np.vstack([cal_bin3[:,0:9], test_bin3[:,0:9]]), 
                                  columns=[f'qt{i}pred3' for i in range(1,10)])], axis=1)
cal_bin4, test_bin4, _ = make_bins(data_ct['pred4'][:data_calib.shape[0]].to_numpy(),
                                   data_ct['pred4'][data_calib.shape[0]:].to_numpy(), n_bins=10)
data_ct = pd.concat([data_ct, pd.DataFrame(np.vstack([cal_bin4[:,0:9], test_bin4[:,0:9]]), 
                                  columns=[f'qt{i}pred4' for i in range(1,10)])], axis=1)

data_ct['pred3sq'] = data_ct['pred3']**2
data_ct['pred4sq'] = data_ct['pred4']**2
data_ct['over'] = 1 * (data_ct['pred']>0.42) 

over = data_ct['over'].to_numpy()[:, None]
mdl3 = data_ct[[f'mdl3_{i}' for i in range(256)]].to_numpy()
mdl4 = data_ct[[f'mdl4_{i}' for i in range(256)]].to_numpy()
mdl3_over = pd.DataFrame(over * mdl3, columns=[f'mdl3_over_{i}' for i in range(256)])
mdl4_over = pd.DataFrame(over * mdl4, columns=[f'mdl4_over_{i}' for i in range(256)])
data_ct = pd.concat([data_ct, mdl3_over, mdl4_over], axis=1)
 
hat_mu_calib = (np.array(data_calib['pred3']) + np.array(data_calib['pred4']))/2
y_calib = np.array(data_calib['Label']) 

hat_mu_test = (np.array(data_test['pred3']) + np.array(data_test['pred4']))/2
y_test = np.array(data_test['Label'])   

### Load the customized features into feature provider

In [12]:
prov = FeaturesProvider(
    f_calib=data_cal_pred['pred'].values,
    f_test=data_test_pred['pred'].values,
    E_calib=calib_feature_all.values,
    E_test=test_feature_all.values,
)
soft_all = data_ct[['pred'+str(x) for x in range(8) if x not in [3,4]]  + 
                   ['qt'+str(x)+'pred4' for x in np.arange(1,10)] + ['pred3sq', 'pred4sq'] + 
                   ['qt'+str(x)+'pred3' for x in np.arange(1,10)] + 
                   ['mdl3_over_' + str(x) for x in range(256)] + ['mdl4_' + str(x) for x in range(256)]]
prov.set_soft_block(
    phi_calib = soft_all.iloc[range(data_calib.shape[0]), :].values,
    phi_test = soft_all.iloc[range(data_calib.shape[0], data_ct.shape[0]), :].values
)
prov.set_force_block(
    F_calib = data_ct[['pred3', 'pred4', 'over' ]].iloc[range(data_calib.shape[0]), :].values,
    F_test = data_ct[['pred3', 'pred4', 'over' ]].iloc[range(data_calib.shape[0], data_ct.shape[0]), :].values
)
prov.set_backup_block(
    Fbc = [data_ct[['pred3', 'pred4', 'over' ]].iloc[range(data_calib.shape[0]), :].values, 
           data_ct[['over']].iloc[range(data_calib.shape[0]), :].values],
    Fbt = [data_ct[['pred3', 'pred4', 'over' ]].iloc[range(data_calib.shape[0], data_ct.shape[0]), :].values, 
           data_ct[['over']].iloc[range(data_calib.shape[0], data_ct.shape[0]), :].values]
)

### Fit the weights using TxConformal and construct p-values

In [13]:
# fit TxConformal to get weights
txc = TxConformal(score_name="clip", 
                  grid = [(x,y) for x in (0.1, 0.2, 0.5, 1) for y in (0.00001, 0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1)],
                  tolerances=[1e-4],
                  alpha=0.2, cutoff=0.5)
txc.fit(prov, data_calib['Label'].values, print_level=-1)

# extrapolate weights to test set
xcal_used = txc._meta['eb_meta']['calib_x'].copy()
xtest_used = txc._meta['eb_meta']['test_x'].copy()
w_calib = txc._weights_calib 
mdl = LinearRegression().fit(xcal_used, np.log(w_calib)) 
w_test_pred = np.exp(mdl.predict(xtest_used))

# rank by weighted p-value 
pval_test = [ (np.sum(w_calib * (hat_mu_calib >= hat_mu_test[j]) * (1-data_calib['Label'])) + w_test_pred[j]) / (np.sum(w_calib) + w_test_pred[j]) for j in range(len(w_test_pred))]
df_pval = pd.DataFrame({"index":  data_test['index'], "pred": hat_mu_test, "weight": w_test_pred, "pval": pval_test}).sort_values(by = 'pval', ascending=True)

### Rank by conformal p-values and make selections

In [14]:
# select top 100 compounds by p-value
df_sel = df_pval.iloc[0:100,:]
df_sel

,index,pred,weight,pval
3439,32007,0.643275,0.084534,0.000093
13340,184222,0.741167,0.097862,0.000097
1381,4141,0.650039,0.190854,0.000162
5273,237768,0.795315,0.274703,0.000202
1889,217165,0.779778,0.390174,0.000277
...,...,...,...,...
10820,47947,0.449474,4.963894,0.005243
13342,165337,0.449851,4.986094,0.005257
12712,274641,0.417082,2.287900,0.005291
12392,339064,0.462960,5.540981,0.005331


In [18]:
# point estimate
fdp_est = len(hat_mu_test) * np.max(df_sel['pval']) 
print(f"Estimated number of false positives: {fdp_est:.2f}")

# standard deviation 
std_cal = np.std(w_calib* (1*(hat_mu_calib >=0.48)-100/len(y_test)) ) * len(y_test) / 100 /np.sqrt(len(y_calib))
std_test = np.sqrt(fdp_est * 100 * (1- fdp_est *100/14000))
std_both = np.sqrt(std_cal**2 + std_test**2)
ci_low = fdp_est - 1.6 * std_both
ci_high = fdp_est + 1.6 * std_both
print(f"90% CI: ({ci_low:.2f}, {ci_high:.2f})")

Estimated number of false positives: 80.30
90% CI: (-13.33, 173.93)
